# Variational classifier

Train a tiny parity classifier with parameter-shift gradients and compare predictions and loss.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

A variational classifier encodes classical features, evaluates a trainable QNode, and updates parameters from labeled data.

In [2]:
features = np.asarray([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
labels = np.asarray([1.0, -1.0, -1.0, 1.0])

def make_model(device):
    @qml.qnode(device, diff_method="parameter-shift")
    def circuit(x, weights):
        qml.RX(np.pi * x[0], wires=0)
        qml.RX(np.pi * x[1], wires=1)
        qml.RY(weights[0], wires=0)
        qml.RY(weights[1], wires=1)
        qml.CNOT(wires=[0, 1])
        qml.RY(weights[2], wires=1)
        return qml.expval(qml.Z(1))
    return circuit

def train(model):
    weights = pnp.array([0.2, -0.1, 0.3], requires_grad=True)
    def loss(current):
        predictions = pnp.stack([model(row, current) for row in features])
        return pnp.mean((predictions - labels) ** 2)
    trace = []
    for _ in range(6):
        trace.append(float(loss(weights)))
        weights = weights - 0.18 * qml.grad(loss)(weights)
    predictions = np.asarray([model(row, weights) for row in features], dtype=float)
    return np.asarray(trace), predictions

reference_model = make_model(qml.device("default.qubit", wires=2))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(lambda: train(reference_model), repeats=2)

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_model = make_model(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: train(mettleq_model), repeats=2)
trace_error = max_abs_error(reference[0], candidate[0])
prediction_error = max_abs_error(reference[1], candidate[1])
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

Training losses and final predictions are compared, not only classification accuracy.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/08_variational_classifier.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="training trace and predictions atol=8e-5",
    passed=max(trace_error, prediction_error) <= 8e-5,
    exact_match=bool(np.array_equal(reference[1], candidate[1])),
    selected_method=method,
    selected_device=device,
    metrics={"trace_error": trace_error, "prediction_error": prediction_error, "predictions": candidate[1]},
)


Comparison summary
------------------
Correctness contract: PASS — training trace and predictions atol=8e-5
SDK reference median: 133.507 ms
MettleQ median:       291.579 ms
Timing interpretation: the SDK reference was 2.184x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "training trace and predictions atol=8e-5", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"prediction_error": 1.1114630804609504e-07, "predictions": [0.9678730368614197, -0.9678730368614197, -0.9296988844871521, 0.9296988844871521], "trace_error": 1.565388414812713e-08}, "mettleq_median_ms": 291.5786875091726, "notebook": "pennylane/08_variational_classifier.ipynb", "notes": "", "passed": true, "python": "3.13.2", "reference_median_ms": 133.50700000592042, "reference_over_mettleq": 0.4578764008659601, "schema_version

## What should you conclude?

For tiny datasets the Python training loop dominates. Device acceleration matters when circuits or batches grow.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.